<!--    
    ================================================================================================
    
    Copyright (c) 2026 Ilya Snegov

    Licensed under the Apache License, Version 2.0 (the "License");
    you may not use this file except in compliance with the License.
    You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

    ================================================================================================
    notebooks/03-model-development.ipynb
    ================================================================================================
-->

# **Model Development**

*A Jupyter notebook for training an XGBoost regressor and performing hyperparameter optimization via Optuna on the training and validation datasets.*

## **I. Setup the environment**

### **Import required libraries**

In [ ]:
import os

import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from scipy.stats import spearmanr
import optuna
import rustuna
from rustuna.converter import ToOptunaSampler

### **Load the train and validation datasets**

In [ ]:
preprocessed_train_df = pd.read_parquet("./data/preprocessed_train.parquet")
preprocessed_valid_df = pd.read_parquet("./data/preprocessed_valid.parquet")

### **Define configuration parameters and extract the feature matrices and target vectors**

In [ ]:
SEED = 42  # Random seed ensuring reproducibility in model training and hyperparameter sampling

N_JOBS = os.cpu_count() // 4    # Number of CPU cores allocated for parallel processing, 
                                # calculated as one-fourth of the total available cores 
                                # to prevent resource oversubscription

N_TRIALS = 200              # Total number of optimization trials for the hyperparameter search space
N_STARTUP_TRIALS = 20       # Number of initial random trials before the TPE sampler begins exploiting the search space

EARLY_STOPPING_ROUNDS = 50  # Maximum number of consecutive rounds without validation score improvement before halting training

# Partition the preprocessed datasets into feature matrices and target vectors for model training and evaluation
X_TRAIN = preprocessed_train_df.drop(columns=['target'])    # Feature matrix for the training set
Y_TRAIN = preprocessed_train_df['target']                   # Target vector for the training set

X_VALID = preprocessed_valid_df.drop(columns=['target'])    # Feature matrix for the validation set
Y_VALID = preprocessed_valid_df['target']                   # Target vector for the validation set



STUDY_NAME = "xgb_hyperparameter_optimization"  # Unique identifier for the optimization study, enabling persistent tracking and 
                                                # retrieval of trial history across execution sessions

STORAGE_PATH = "./data/optuna_study.db"         # Local filesystem path for the SQLite database to persist trial results, 
                                                # hyperparameter configurations, and evaluation metrics

STORAGE_URL = f"sqlite:///{STORAGE_PATH}"       # SQLAlchemy-compliant connection string required by Optuna to establish a reliable 
                                                # backend connection to the persistent storage

### **Define the objective function for hyperparameter optimization**

**Justification of the loss function and internal evaluation metrics**

The standard Huber loss ($\mathcal{L}_{Huber}$) is intentionally replaced by its smooth approximation, the Pseudo-Huber loss (`reg:pseudohubererror`). XGBoost relies on the Newton-Raphson method for tree construction, which strictly requires a twice-differentiable objective function to compute the Hessian. The standard Huber loss is non-differentiable at the transition threshold, whereas the Pseudo-Huber loss provides the necessary continuous first and second derivatives while preserving robustness against extreme outliers.

Furthermore, the Spearman Rank Correlation (Information Coefficient) cannot be utilized as XGBoost's internal evaluation metric for early stopping. Spearman is a non-differentiable, rank-based statistic that is mathematically incompatible with gradient-based tree splitting and internal stopping mechanisms. Consequently, Root Mean Squared Error (RMSE) is employed strictly as a proxy for XGBoost's internal early stopping to prevent overfitting. The Spearman correlation is computed externally and returned to the optimization framework, effectively decoupling the internal gradient descent mechanics from the ultimate ranking objective.

In [ ]:
def objective(trial: optuna.trial.Trial) -> float:
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 1e-8, 1.0, log=True),
    }

    model = XGBRegressor(
        objective="reg:pseudohubererror",
        eval_metric="rmse",
        random_state=SEED,
        n_jobs=N_JOBS,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        **params
    )

    model.fit(
        X_TRAIN,
        Y_TRAIN,
        eval_set=[(X_VALID, Y_VALID)],
        verbose=False,
    )

    y_pred = model.predict(X_VALID)

    # Reject diverged or degenerate configurations before computing the rank statistic
    if not np.isfinite(y_pred).all() or np.unique(y_pred).size <= 1:
        return -1.0

    spearman_corr = spearmanr(Y_VALID, y_pred).statistic

    if not np.isfinite(spearman_corr):
        return -1.0

    return spearman_corr

### **Initialize the persistent relational database storage backend**

In [ ]:
storage = optuna.storages.RDBStorage(
    url=STORAGE_URL,
    engine_kwargs={"connect_args": {"check_same_thread": False}}
)

### **Initialize the Tree-structured Parzen Estimator sampler for Bayesian optimization**

In [ ]:
rustuna_sampler = rustuna.samplers.TPESampler(
    seed=SEED,
    n_startup_trials=N_STARTUP_TRIALS
)
sampler = ToOptunaSampler(rustuna_sampler)

### **Initialize the hyperparameter optimization study**

In [ ]:
study = optuna.create_study(
    study_name=STUDY_NAME,
    direction="maximize",
    sampler=sampler,
    storage=storage,
    load_if_exists=True
)

print(f"Study '{STUDY_NAME}' initialized. Total trials so far: {len(study.trials)}")

## **II. Execute the hyperparameter optimization study**

In [ ]:
study.optimize(objective, n_trials=N_TRIALS)

## **III. Extract the optimal hyperparameters**

In [ ]:
best_trial = study.best_trial
best_params = best_trial.params

print(f"Best trial number: {best_trial.number}")
print(f"Best trial value:  {best_trial.value:.6f}")
print("\nBest hyperparameters:")
print(best_params)

## **IV. Train the final XGBRegressor with optimized hyperparameters**

In [ ]:
X_TRAIN_FINALE = pd.concat([X_TRAIN, X_VALID], axis=0)
Y_TRAIN_FINALE = pd.concat([Y_TRAIN, Y_VALID], axis=0)

model = XGBRegressor(**best_params)
model.fit(X_TRAIN_FINALE, Y_TRAIN_FINALE, verbose=False)

## **V. Serialize the trained model to JSON format**

In [ ]:
model.save_model("./data/finml_gbdt_regressor.json")